## Setup & Annotated Corpus

In [4]:
import re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod5", exist_ok = True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
try:
    import spacy
    try:
        nlp_base = spacy.load("en_core_web_sm")
    except OSError:
        os.system("python -m spacy download en_core_web_sm -q")
        nlp_base = spacy.load("en_core_web_sm")
    SPACY_OK = True
    print(f"spaCy {spacy.__version__} loaded")
except ImportError:
    SPACY_OK = False
    print("spaCy not available - using regex NER fallback")

# Annotated clinical texts
ANNOTATED = [
    {"id":"T001","text":
     "The patient has type 2 diabetes mellitus managed with metformin 1000 mg twice daily "
     "and lisinopril 10 mg for hypertension. She also takes furosemide 40 mg for heart failure. "
     "On exam: bilateral pitting edema to the knees, S3 gallop present.",
     "gold":[("type 2 diabetes mellitus","DISEASE"),("metformin","DRUG"),
             ("lisinopril","DRUG"),("hypertension","DISEASE"),("furosemide","DRUG"),
             ("heart failure","DISEASE"),("bilateral pitting edema","FINDING"),
             ("S3 gallop","FINDING")]},
    {"id":"T002","text":
     "CT chest revealed a 5mm pulmonary nodule in the right lower lobe and bilateral pleural effusions. "
     "No pulmonary embolism identified. The patient was started on rivaroxaban 20 mg for atrial fibrillation. "
     "Echo showed mitral regurgitation and aortic stenosis.",
     "gold":[("pulmonary nodule","FINDING"),("right lower lobe","ANATOMY"),
             ("bilateral pleural effusions","FINDING"),("pulmonary embolism","DISEASE"),
             ("rivaroxaban","DRUG"),("atrial fibrillation","DISEASE"),
             ("mitral regurgitation","DISEASE"),("aortic stenosis","DISEASE")]},
    {"id":"T003","text":
     "Post-op day 2 following coronary artery bypass grafting. Chest pain at sternotomy site rated 6/10. "
     "No wound infection. Warfarin 5 mg initiated for mechanical aortic valve. "
     "Creatinine elevated at 1.8, likely acute kidney injury.",
     "gold":[("coronary artery bypass grafting","PROCEDURE"),("chest pain","SYMPTOM"),
             ("sternotomy site","ANATOMY"),("wound infection","DISEASE"),
             ("Warfarin","DRUG"),("acute kidney injury","DISEASE")]},
    {"id":"T004","text":
     "Impression: Community-acquired pneumonia with sepsis. "
     "Started ceftriaxone 1g IV daily and azithromycin 500 mg IV daily. Blood cultures pending. "
     "CBC: WBC 18.4, Hgb 9.2. CXR: right lower lobe consolidation.",
     "gold":[("Community-acquired pneumonia","DISEASE"),("sepsis","DISEASE"),
             ("ceftriaxone","DRUG"),("azithromycin","DRUG"),
             ("right lower lobe consolidation","FINDING")]},
]
print(f"Annoted corpus: {len(ANNOTATED)} texts")
print(f"Total gold entities: {sum(len(t['gold']) for t in ANNOTATED)}")

spaCy 3.8.15 loaded
Annoted corpus: 4 texts
Total gold entities: 27


In [2]:
%pip install spacy

   ---------------------------------------- 0.0/14.3 MB ? eta -:--:--
   ------------------- -------------------- 6.8/14.3 MB 36.3 MB/s eta 0:00:01
   ---------------------------------------  14.2/14.3 MB 36.9 MB/s eta 0:00:01
   ---------------------------------------- 14.3/14.3 MB 31.7 MB/s  0:00:00
   ---------------------------------------- 0.0/650.8 kB ? eta -:--:--
   ---------------------------------------- 650.8/650.8 kB 13.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 21.8 MB/s  0:00:00
   ---------------------------------------- 0.0/6.2 MB ? eta -:--:--
   ---------------------------------------- 6.2/6.2 MB 34.0 MB/s  0:00:00

   ----------------------------------------  0/15 [wasabi]
   ----------------------------------------  0/15 [wasabi]
   ----------------------------------------  0/15 [wasabi]
   -- -------------------------------------  1/15 [spacy-loggers]
   -- ------------

In [8]:
DISEASE_PATTERNS = [
    "diabetes mellitus","type 2 diabetes mellitus","type 1 diabetes","DM2",
    "hypertension","heart failure","congestive heart failure","ADHF",
    "COPD","chronic obstructive pulmonary disease","asthma","pneumonia","CAP",
    "atrial fibrillation","AFib","coronary artery disease","CAD",
    "myocardial infarction","STEMI","NSTEMI","ACS",
    "chronic kidney disease","CKD","acute kidney injury","AKI","ESRD",
    "pulmonary embolism","PE","deep vein thrombosis","DVT",
    "sepsis","severe sepsis","septic shock","bacteremia",
    "stroke","TIA","ischemic stroke",
    "mitral regurgitation","aortic stenosis","aortic regurgitation",
    "pleural effusion","bilateral pleural effusions","pneumothorax",
    "community-acquired pneumonia","wound infection","cellulitis",
]
DRUG_PATTERNS = [
    "metformin","lisinopril","furosemide","warfarin","rivaroxaban","apixaban",
    "aspirin","clopidogrel","ticagrelor","heparin","enoxaparin",
    "ceftriaxone","azithromycin","vancomycin","piperacillin-tazobactam","meropenem",
    "metoprolol","carvedilol","atorvastatin","rosuvastatin",
    "amiodarone","digoxin","diltiazem","prednisone","methylprednisolone","dexamethasone",
    "albuterol","ipratropium","tiotropium","fluticasone",
    "insulin","metformin","empagliflozin","liraglutide","semaglutide",
    "omeprazole","pantoprazole","ondansetron","morphine","oxycodone","acetaminophen",
    "nitroglycerin","norepinephrine","sacubitril","spironolactone",
]
PROCEDURE_PATTERNS = [
    "coronary artery bypass grafting","CABG","percutaneous coronary intervention","PCI",
    "cardiac catheterization","echocardiogram","ECHO","transesophageal echocardiogram",
    "CT chest","CT abdomen","CT head","CXR","chest X-ray","MRI brain",
    "bronchoscopy","thoracentesis","paracentesis","lumbar puncture",
    "colonoscopy","endoscopy","mechanical ventilation","intubation","hemodialysis",
]
ANATOMY_PATTERNS = [
    "right lower lobe","left lower lobe","right upper lobe","left upper lobe",
    "sternotomy site","thoracotomy site","left ventricle","right ventricle",
    "mitral valve","aortic valve","aorta","pulmonary artery",
    "liver","spleen","kidney","gallbladder","pancreas","colon",
]
SYMPTOM_PATTERNS = [
    "chest pain","dyspnea","shortness of breath","SOB","cough","productive cough",
    "fever","chills","nausea","vomiting","abdominal pain","headache",
    "dizziness","syncope","palpitations","fatigue","weakness","edema",
    "hemoptysis","confusion","altered mental status",
]
FINDING_PATTERNS = [
    "bilateral pitting edema","pitting edema","S3 gallop","S4 gallop","murmur",
    "crackles","rales","wheezes","rhonchi","decreased breath sounds",
    "consolidation","air bronchograms","ground glass opacity","atelectasis",
    "ST elevation","ST depression","T-wave inversion","pulmonary nodule",
    "right lower lobe consolidation","bilateral pleural effusions","cardiomegaly",
    "right ventricular enlargement","hepatomegaly","splenomegaly","jaundice",
]

ENTITY_LISTS = {
    "DISEASE":DISEASE_PATTERNS,"DRUG":DRUG_PATTERNS,"PROCEDURE":PROCEDURE_PATTERNS,
    "ANATOMY":ANATOMY_PATTERNS,"SYMPTOM":SYMPTOM_PATTERNS,"FINDING":FINDING_PATTERNS,
}

total = sum(len(v) for v in ENTITY_LISTS.values())
print(f"Total clinical patterns: {total}")
for label, patterns in ENTITY_LISTS.items():
    print(f" {label:12s}: {len(patterns):3d} patterns")

Total clinical patterns: 178
 DISEASE     :  46 patterns
 DRUG        :  45 patterns
 PROCEDURE   :  23 patterns
 ANATOMY     :  18 patterns
 SYMPTOM     :  21 patterns
 FINDING     :  25 patterns


## spaCy EntityRuler Pipeline

In [11]:
if SPACY_OK:
    from spacy.pipeline import EntityRuler
    clinical_nlp = spacy.blank("en")
    clinical_nlp.add_pipe("sentencizer")
    ruler = clinical_nlp.add_pipe("entity_ruler", config={"overwrite_ents": True})
    ruler_patterns = []
    for label, patterns in ENTITY_LISTS.items():
        for p in patterns:
            ruler_patterns.append({"label":label, "pattern":p})
            ruler_patterns.append({"label":label,"pattern":p.lower()})
            ruler_patterns.append({"label":label,"pattern":p.title()})
    ruler.add_patterns(ruler_patterns)
    print(f"EntityRuler: {len(ruler_patterns)} patterns loaded")

    doc = clinical_nlp(ANNOTATED[0]["text"])
    print(f"\nNER on T001 ({len(list(doc.ents))} entites):")
    for ent in doc.ents:
        print(f" [{ent.label_:12s}] '{ent.text}'")

EntityRuler: 534 patterns loaded

NER on T001 (8 entites):
 [DISEASE     ] 'type 2 diabetes mellitus'
 [DRUG        ] 'metformin'
 [DRUG        ] 'lisinopril'
 [DISEASE     ] 'hypertension'
 [DRUG        ] 'furosemide'
 [DISEASE     ] 'heart failure'
 [FINDING     ] 'bilateral pitting edema'
 [FINDING     ] 'S3 gallop'


## Regex Fallback NER

In [12]:
def regex_ner(text, entity_lists):
    entities = []
    for label, patterns in entity_lists.items():
        for pattern in patterns:
            regex = r'\b'+re.escape(pattern)+r'\b'
            for m in re.finditer(regex, text, re.IGNORECASE):
                entities.append({"text":m.group(), "label":label, "start":m.start(),"end":m.end()})
    #Remove overlapping - keep longer match
    entities.sort(key=lambda x: (x["start"], -(x["end"]-x["start"])))
    filtered = []; prev_end = -1
    for ent in entities:
        if ent["start"] >= prev_end:
            filtered.append(ent); prev_end = ent["end"]
    return filtered
        